# 05 - Jetson Orin Nano Deployment Export

**What we know from training (04_lightweight_dl_model.ipynb):**
- Best model : unidirectional GRU, `hidden_size=64`, max-pooling over all 64 timesteps
- F1=0.264, Precision=0.232, Recall=0.306, ROC-AUC=0.635 (vs IsolationForest baseline F1=0.237)
- Best threshold : 0.65
- Checkpoint : `models/trained/gru_best.pt` + `models/trained/gru_config.json`

**Objectives:**
- Rebuild the GRU architecture from `gru_config.json` and load the trained weights
- Export to ONNX with a dynamic batch axis
- Verify the ONNX model's output matches the original PyTorch model
- Benchmark CPU inference latency here as a reference number only (not the deployment target)
- Package a self-contained bundle (ONNX model + scaler + deployment config) for USB transfer

**Why export to ONNX rather than deploy PyTorch directly:**
- ONNX is a hardware-agnostic intermediate format (the same exported file can run via ONNX Runtime's CPU, CUDA or TensorRT execution providers without touching the model code again)
- The Jetson's TensorRT stack consumes ONNX rather than raw PyTorch checkpoints (this is the standard edge-deployment path, not a workaround)
- Exporting and verifying correctness here, before anything touches the Jetson, catches bugs early when they're cheap to fix

[Nvidia jetson orin nano quick start guide](https://docs.nvidia.com/jetson/orin-nano-devkit/user-guide/latest/quick_start.html) \
[Tutorial for AI models into jetson orin nano board](https://forums.developer.nvidia.com/t/ai-models-that-run-on-jetson-orin-nano-super-8gb-a-practical-guide/365412) \
[Video : How to run model in NVIDIA jetson orin nano board](https://www.youtube.com/watch?v=lWEdgNX2uWk) \
[Video : Getting started with NVIDIA jetson orin nano board](https://www.youtube.com/watch?v=NAdXA2IsZAM)


---
## Imports & Paths

In [1]:
import json
import time
import shutil
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import onnx
import onnxruntime as ort

ROOT = Path("..")
MODELS_TRAINED = ROOT / "models" / "trained"
MODELS_BASELINE = ROOT / "models" / "baseline"
MODELS_ONNX = ROOT / "models" / "onnx"
DATA_PROC = ROOT / "data"   / "processed"

MODELS_ONNX.mkdir(parents=True, exist_ok=True)

for p in [MODELS_TRAINED / "gru_best.pt",
          MODELS_TRAINED / "gru_config.json",
          MODELS_BASELINE / "scaler_starter_channels.pkl"]:
    status = "File exist : " if p.exists() else "File missing : "
    print(f"{status}  {p}")

print(f"\nONNX version         : {onnx.__version__}")
print(f"ONNX Runtime version : {ort.__version__}")

File exist :   ../models/trained/gru_best.pt
File exist :   ../models/trained/gru_config.json
File exist :   ../models/baseline/scaler_starter_channels.pkl

ONNX version         : 1.22.0
ONNX Runtime version : 1.23.2


---
## Load Model Config & Checkpoint

`gru_config.json` records the exact architecture and training result of the checkpoint we're about to export

In [2]:
with open(MODELS_TRAINED / "gru_config.json") as f:
    model_config = json.load(f)

print(json.dumps(model_config, indent=2))

print(f"\nModel to export :")
print(f"  Architecture                      : GRU (bidirectional={model_config['bidirectional']})")
print(f"  Hidden size                       : {model_config['hidden_size']}")
print(f"  Window size                       : {model_config['window_size']}")
print(f"  Channels                          : {model_config['starter_cols']}")
print(f"  Best threshold                    : {model_config['best_threshold']:.4f}")
print(f"  Validation F1                     : {model_config['val_metrics']['f1']:.4f}")
print(f"  vs IsolationForest baseline (ΔF1) : {model_config['vs_baseline_iso']['f1_delta']:+.4f}")

{
  "architecture": "GRU",
  "bidirectional": false,
  "seed": 42,
  "n_features": 6,
  "hidden_size": 64,
  "num_layers": 1,
  "dropout": 0.2,
  "window_size": 64,
  "starter_cols": [
    "channel_41",
    "channel_42",
    "channel_43",
    "channel_44",
    "channel_45",
    "channel_46"
  ],
  "best_threshold": 0.6500000000000001,
  "training": {
    "n_epochs": 40,
    "batch_size": 256,
    "lr": 0.001,
    "weight_decay": 0.0001,
    "pos_weight": 8.46627665878681,
    "patience": 10,
    "epochs_ran": 32,
    "best_val_f1": 0.21593781657404165,
    "best_val_pr_auc": 0.26523123005082055,
    "subsample_ratio": 0.05
  },
  "val_metrics": {
    "f1": 0.26380868865026447,
    "precision": 0.2317771047849788,
    "recall": 0.30611356785849136,
    "roc_auc": 0.635483877925912
  },
  "vs_baseline_iso": {
    "f1_delta": 0.02726003841058741,
    "precision_delta": 0.05563638257545475,
    "recall_delta": -0.05390409512412081,
    "roc_auc_delta": 0.029042579541409985
  }
}

Model to 

---
## Rebuild Model Architecture & Load Weights

Same `GRUAnomalyDetector` class as notebook 04, rebuilt here standalone so this notebook doesn't depend on re-running 04. Architecture parameters come from `model_config`

In [3]:
class GRUAnomalyDetector(nn.Module):
    """
    Lightweight GRU (optionally bidirectional) for binary anomaly detection on telemetry windows
    """
    def __init__(self,
                 n_features: int = 6,
                 hidden_size: int = 64,
                 num_layers: int = 2,
                 dropout: float = 0.3,
                 bidirectional: bool = False):
        super().__init__()

        num_directions = 2 if bidirectional else 1

        self.gru = nn.GRU(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional,
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * num_directions, 32),   # num_directions = 2 means forward + backward directions concatenated
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        # x (input shape) : (batch, seq_len, n_features)
        output, _ = self.gru(x)             # (batch, seq_len, hidden_size * num_directions)
        pooled, _ = output.max(dim=1)       # (batch, hidden_size * num_directions) => max over the 64 timesteps
        logits = self.classifier(pooled)
        return logits.squeeze(1)            # (batch,)

model = GRUAnomalyDetector(
    n_features=model_config["n_features"],
    hidden_size=model_config["hidden_size"],
    num_layers=model_config["num_layers"],
    dropout=model_config["dropout"],
    bidirectional=model_config["bidirectional"],
)
model.load_state_dict(torch.load(MODELS_TRAINED / "gru_best.pt", map_location="cpu")) # cpu because no train needed anymore
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"\nTrainable parameters : {n_params:,}")
print(f"Loaded from          : gru_best.pt")


GRUAnomalyDetector(
  (gru): GRU(6, 64, batch_first=True)
  (classifier): Sequential(
    (0): Linear(in_features=64, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=32, out_features=1, bias=True)
  )
)

Trainable parameters : 15,937
Loaded from          : gru_best.pt


---
## Export to ONNX

`window_size` (64) and `n_features` (6) stay fixed. The batch dimension is dynamic via `dynamic_axes` so the exported model can score a single window in real time (batch=1) or a whole batch (for example re-scoring a backlog after a comms outage)

[Using ONNX library](https://medium.com/@vigneshkumar25/onnx-explained-simply-how-to-run-ai-models-anywhere-83706fd9866e) \
[DataCamp - ONNX: Train in Any Framework, Deploy on Any Hardware](https://www.datacamp.com/tutorial/onnx)

In [4]:
WINDOW_SIZE = model_config["window_size"]
N_FEATURES  = model_config["n_features"]

dummy_input = torch.randn(1, WINDOW_SIZE, N_FEATURES, dtype=torch.float32)

onnx_path = MODELS_ONNX / "gru_anomaly_detector.onnx"

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    input_names=["telemetry_window"],
    output_names=["anomaly_logit"],
    dynamic_axes={"telemetry_window": {0: "batch_size"}, "anomaly_logit": {0: "batch_size"}},
    opset_version=18,            # exporter needs >=18 for this model's ops because downgrading to 17 fails
)

onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)

print(f"ONNX model exported and structurally validated : {onnx_path}")
print(f"File size : {onnx_path.stat().st_size / 1e3:.1f} KB")
print(f"Input     : 'telemetry_window'  shape=(batch, {WINDOW_SIZE}, {N_FEATURES})")
print(f"Output    : 'anomaly_logit'     shape=(batch,) - raw logit, apply sigmoid + threshold downstream")


/var/folders/v1/8nl4hjrd2p7_yl1_6hfnt2hw0000gn/T/ipykernel_48518/2374940387.py:8: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0715 20:43:10.027000 48518 site-packages/torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::nms
W0715 20:43:10.028000 48518 site-packages/torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::roi_align
W0715 20:43:10.029000 48518 site-packages/torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::roi_pool
W0715 20:43:10.029000 48518 site-packages/torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::deform_conv2d


[torch.onnx] Obtain model graph for `GRUAnomalyDetector([...]` with `torch.export.export(..., strict=False)`...


/Users/photoli93/Desktop/Projets perso Python/esa_fake_or_real/.conda/sat-anom-esa-adb/lib/python3.10/contextlib.py:142: UserWarning: The tensor attributes self.gru._flat_weights[0], self.gru._flat_weights[1], self.gru._flat_weights[2], self.gru._flat_weights[3] were assigned during export. Such attributes must be registered as buffers using the `register_buffer` API (https://pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.register_buffer).
  next(self.gen)


[torch.onnx] Obtain model graph for `GRUAnomalyDetector([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/Users/photoli93/Desktop/Projets perso Python/esa_fake_or_real/.conda/sat-anom-esa-adb/lib/python3.10/copyreg.py:101: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
ONNX model exported and structurally validated : ../models/onnx/gru_anomaly_detector.onnx
File size : 31.5 KB
Input     : 'telemetry_window'  shape=(batch, 64, 6)
Output    : 'anomaly_logit'     shape=(batch,) - raw logit, apply sigmoid + threshold downstream


---
## Sanity check 1 - between PyTorch model and ONNX model

`onnx.checker.check_model` only confirms the file is structurally valid, not that it computes the same function. We run real telemetry windows through both models and compare outputs directly

In [5]:
# PyTorch model
X_test = np.load(DATA_PROC / "X_test_starter.npy")
sample_idx = np.random.default_rng(42).choice(len(X_test), size=256, replace=False)
X_sample = X_test[sample_idx].astype(np.float32)

with torch.no_grad():
    torch_logits = model(torch.tensor(X_sample)).numpy()

# ONNX model
sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
onnx_logits = sess.run(None, {"telemetry_window": X_sample})[0]

# Worth case-difference between the two models 
max_abs_diff = np.abs(torch_logits - onnx_logits).max()
print(f"Max absolute difference (PyTorch vs ONNX logits) : {max_abs_diff:.8f}")
# Ensure that both model values are very close
assert max_abs_diff < 1e-4, "ONNX export does not match the PyTorch model output!"
print("OK : ONNX export produces (near-)identical outputs to the PyTorch model")

# Convert raw logits into prob
torch_probs = torch.sigmoid(torch.tensor(torch_logits)).numpy()
onnx_probs  = 1 / (1 + np.exp(-onnx_logits))

threshold   = model_config["best_threshold"]

torch_preds = (torch_probs >= threshold).astype(int)
onnx_preds  = (onnx_probs  >= threshold).astype(int)

print(f"\nPredictions match : {(torch_preds == onnx_preds).all()}")
print(f"Anomalies flagged : {(onnx_preds == 1).sum()} / {len(X_sample)} sample windows (threshold={threshold:.2f})")


Max absolute difference (PyTorch vs ONNX logits) : 0.00000131
OK : ONNX export produces (near-)identical outputs to the PyTorch model

Predictions match : True
Anomalies flagged : 8 / 256 sample windows (threshold=0.65)


2026-07-15 20:43:13.405 python[48518:15472746] 2026-07-15 20:43:13.403718 [W:onnxruntime:, execution_frame.cc:874 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {256} for output anomaly_logit


---
## Sanity check 2 - CPU Latency Baseline

Confirms the model is small/fast enough to be a plausible edge candidate and gives a point of comparison for the real Jetson number later

In [6]:
def benchmark_latency(session, X, n_warmup=20, n_runs=200):
    input_name = session.get_inputs()[0].name
    for i in range(n_warmup):
        session.run(None, {input_name: X[i % len(X): i % len(X) + 1]})

    latencies = []
    for i in range(n_runs):
        window = X[i % len(X): i % len(X) + 1]
        t0 = time.perf_counter()
        session.run(None, {input_name: window})
        latencies.append((time.perf_counter() - t0) * 1000)  # ms
    return np.array(latencies)

latencies_single = benchmark_latency(sess, X_sample)

print(f"Single-window inference latency :")
print(f"  Mean : {latencies_single.mean():.3f} ms")
print(f"  P50  : {np.percentile(latencies_single, 50):.3f} ms")
print(f"  P95  : {np.percentile(latencies_single, 95):.3f} ms")
print(f"  P99  : {np.percentile(latencies_single, 99):.3f} ms")


Single-window inference latency :
  Mean : 0.091 ms
  P50  : 0.088 ms
  P95  : 0.108 ms
  P99  : 0.149 ms


---
## Package Deployment Bundle

Self-contained folder for a single USB drag-and-drop: the ONNX model, the fitted `StandardScaler` and a deployment-focused config

In [7]:
scaler_src = MODELS_BASELINE / "scaler_starter_channels.pkl"
scaler_dst = MODELS_ONNX / "scaler_starter_channels.pkl"
shutil.copy(scaler_src, scaler_dst)

# Minimum runtime metadata needed to reproduce inference
deployment_config = {
    "model_file"     : onnx_path.name,
    "scaler_file"    : scaler_dst.name,
    "starter_cols"   : model_config["starter_cols"],
    "window_size"    : model_config["window_size"],
    "n_features"     : model_config["n_features"],
    "best_threshold" : model_config["best_threshold"],
    "val_metrics"    : model_config["val_metrics"],
}
with open(MODELS_ONNX / "deployment_config.json", "w") as f:
    json.dump(deployment_config, f, indent=2)

print("Deployment bundle ready in models/onnx/ :")
for p in sorted(MODELS_ONNX.iterdir()):
    print(f"  {p.name}  ({p.stat().st_size / 1e3:.1f} KB)")

Deployment bundle ready in models/onnx/ :
  deployment_config.json  (0.5 KB)
  gru_anomaly_detector.onnx  (31.5 KB)
  gru_anomaly_detector.onnx.data  (63.5 KB)
  scaler_starter_channels.pkl  (0.8 KB)


Copy this entire folder (`satellite_anomaly_esa_adb/models/onnx`) to the Jetson orion nano board via USB